# Mission EdStats : qualité des données

Mark veut savoir si les données EdStats peuvent guider l'expansion internationale d'academy : quels pays cibler, et comment ce potentiel évolue.

Avant de chercher des pays, on vérifie les 5 fichiers, un par un : clés, doublons, valeurs manquantes, colonnes à garder. Rien n'est modifié ici, la phase de nettoyage agira ensuite.


## Mise en place

### 1. Imports et dossier des données

Où se trouvent les 5 CSV à analyser ?


In [1]:
import pandas as pd
from pathlib import Path

candidates = [
    Path.cwd() / "data" / "input",        # lancé depuis la racine du projet
    Path.cwd() / ".." / "data" / "input",    # lancé depuis le dossier notebooks/
]

INPUT_DIR = None
for candidate in candidates:
    if candidate.exists():
        INPUT_DIR = candidate.resolve()  # chemin absolu
        break

if INPUT_DIR is None:
    raise FileNotFoundError("Dossier data/input/ introuvable")

INPUT_DIR

PosixPath('/Users/emmanuel/Formation Data Engineer/Projets/P2 - Analyse données systèmes éducatifs/data/input')

### 2. Chargement brut

Comment lire chaque CSV brut, tel quel, en écartant la colonne vide « Unnamed » (sans nom) que crée la virgule finale de chaque ligne ?


In [2]:
def load_csv_analysis(name):
    """Lit le CSV `name` depuis INPUT_DIR, tel quel (brut), pour l'analyse."""
    return pd.read_csv(
        INPUT_DIR / name,
        encoding="utf-8",
        usecols=lambda column: "Unnamed" not in column,
    )

### 3. Chargement des 5 CSV

Les 5 fichiers se chargent-ils sans erreur ?


In [3]:
FICHIERS = [
    "EdStatsCountry.csv",        # référentiel des pays et agrégats
    "EdStatsSeries.csv",         # dictionnaire des indicateurs
    "EdStatsData.csv",           # mesures par pays, indicateur et année
    "EdStatsCountry-Series.csv", # sources par couple pays × indicateur
    "EdStatsFootNote.csv",       # annotations par pays, indicateur et année
]

dfs = {}
for name in FICHIERS:
    dfs[name] = load_csv_analysis(name)
    print(f"{name} : {dfs[name].shape[0]} lignes, {dfs[name].shape[1]} colonnes")

EdStatsCountry.csv : 241 lignes, 31 colonnes
EdStatsSeries.csv : 3665 lignes, 20 colonnes


EdStatsData.csv : 886930 lignes, 69 colonnes
EdStatsCountry-Series.csv : 613 lignes, 3 colonnes
EdStatsFootNote.csv : 643638 lignes, 4 colonnes


### 4. Taille de chaque fichier, vue d'ensemble

Lignes, colonnes, types, mémoire : que contient chaque fichier ?


In [4]:
for name, df in dfs.items():
    print(name)
    df.info(verbose=False)  # résumé compact : lignes, colonnes, types, mémoire

EdStatsCountry.csv
<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Columns: 31 entries, Country Code to Latest water withdrawal data
dtypes: float64(3), str(28)
memory usage: 58.5 KB
EdStatsSeries.csv
<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Columns: 20 entries, Series Code to License Type
dtypes: float64(5), str(15)
memory usage: 572.8 KB
EdStatsData.csv
<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Columns: 69 entries, Country Name to 2100
dtypes: float64(65), str(4)
memory usage: 466.9 MB
EdStatsCountry-Series.csv
<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Columns: 3 entries, CountryCode to DESCRIPTION
dtypes: str(3)
memory usage: 14.5 KB
EdStatsFootNote.csv
<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Columns: 4 entries, CountryCode to DESCRIPTION
dtypes: str(4)
memory usage: 19.6 MB


### Bilan de la mise en place

Les 5 CSV sont chargés bruts, valeurs manquantes comprises, pour être comptées. `EdStatsData.csv` écrase le lot : 886 930 lignes, 466,9 Mo en mémoire.

Chaque section suivante analyse un fichier, en commençant par `EdStatsCountry.csv`.


## EdStatsCountry.csv : le référentiel des pays

*Une ligne = un pays ou un agrégat (ex. « Arab World », le monde arabe).*


### 1. Une ligne = un pays ou un agrégat

La clé `Country Code` joindra les fichiers : est-elle unique ? Aperçu des 3 premières lignes :


In [5]:
dfs["EdStatsCountry.csv"].head(3)  # aperçu brut, via `dfs` réservé à l'affichage

,Country Code,Short Name,Table Name,Long Name,2-alpha code,Currency Unit,Special Notes,Region,Income Group,WB-2 code,...,Government Accounting concept,IMF data dissemination standard,Latest population census,Latest household survey,Source of most recent Income and expenditure data,Vital registration complete,Latest agricultural census,Latest industrial data,Latest trade data,Latest water withdrawal data
0,ABW,Aruba,Aruba,Aruba,AW,Aruban florin,SNA data for 2000-2011 are updated from offici...,Latin America & Caribbean,High income: nonOECD,AW,...,NaN,NaN,2010,NaN,NaN,Yes,NaN,NaN,2012.0,NaN
1,AFG,Afghanistan,Afghanistan,Islamic State of Afghanistan,AF,Afghan afghani,Fiscal year end: March 20; reporting period fo...,South Asia,Low income,AF,...,Consolidated central government,General Data Dissemination System (GDDS),1979,"Multiple Indicator Cluster Survey (MICS), 2010/11","Integrated household survey (IHS), 2008",NaN,2013/14,NaN,2012.0,2000
2,AGO,Angola,Angola,People's Republic of Angola,AO,Angolan kwanza,"April 2013 database update: Based on IMF data,...",Sub-Saharan Africa,Upper middle income,AO,...,Budgetary central government,General Data Dissemination System (GDDS),1970,"Malaria Indicator Survey (MIS), 2011","Integrated household survey (IHS), 2008",NaN,2015,NaN,NaN,2005


In [6]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsCountry.csv")

cle = "Country Code"  # clé métier (PK) d'après le dictionnaire de 00_data_overview.ipynb
print(f"{len(df)} lignes, {df[cle].nunique()} valeurs distinctes de '{cle}'")
df[cle].is_unique  # True = chaque valeur de la clé n'apparaît que sur une seule ligne

241 lignes, 241 valeurs distinctes de 'Country Code'


True

**Lecture** : 241 lignes, 241 codes distincts. Clé unique, la jointure ne créera pas de doublons.


### 2. Taille du fichier

Combien de lignes, combien de colonnes, et de quel type ?


In [7]:
df.shape
df.info()  # en plus de shape : type et remplissage de chaque colonne

<class 'pandas.DataFrame'>
RangeIndex: 241 entries, 0 to 240
Data columns (total 31 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   Country Code                                       241 non-null    str    
 1   Short Name                                         241 non-null    str    
 2   Table Name                                         241 non-null    str    
 3   Long Name                                          241 non-null    str    
 4   2-alpha code                                       238 non-null    str    
 5   Currency Unit                                      215 non-null    str    
 6   Special Notes                                      145 non-null    str    
 7   Region                                             214 non-null    str    
 8   Income Group                                       214 non-null    str    
 9   WB-2 code            

**Lecture** : 241 lignes × 31 colonnes : 31 renseignements par pays ou agrégat.


### 3. Des doublons ?

Le même pays apparaît-il deux fois ?


In [8]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"{nb_doublons} doublon(s) détecté(s), à supprimer en phase de nettoyage avec drop_duplicates()")
else:
    print("aucun doublon : rien à supprimer en phase de nettoyage")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : rien à supprimer en phase de nettoyage


**Lecture** : 0 doublon, aucun pays en double.


### 4. Valeurs manquantes et taux de remplissage

Quelles colonnes sont remplies, et à quel point ?


In [9]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

4 colonne(s) à 100 % de remplissage, 0 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
Country Code,0,0.00,100.00
Table Name,0,0.00,100.00
Long Name,0,0.00,100.00
Short Name,0,0.00,100.00
WB-2 code,1,0.41,99.59
2-alpha code,3,1.24,98.76
System of National Accounts,26,10.79,89.21
Currency Unit,26,10.79,89.21
Region,27,11.20,88.80
Income Group,27,11.20,88.80


**Lecture** : les 4 colonnes d'identification sont complètes, aucune colonne vide ; `Region` (région) et `Income Group` (groupe de revenu) descendent à 88,8 %, le minimum est 13,3 % (`National accounts reference year`). Ce tableau dit sur quelles colonnes s'appuyer pour classer les pays.


### 5. Colonnes inutilisables (à supprimer en phase de nettoyage)

Quelles colonnes ne distinguent pas les pays : vides à 100 %, constantes, ou en double d'une autre ?


In [10]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> rien pour distinguer les lignes
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

# 3) colonnes en double : "WB-2 code" (code interne BM à 2 lettres) reprend "2-alpha code" (code ISO)
sim = (df["2-alpha code"].fillna("<NA>") == df["WB-2 code"].fillna("<NA>")).mean()
print(f"'2-alpha code' == 'WB-2 code' dans {sim:.0%} des lignes")
redondantes = ["WB-2 code"]

a_supprimer = vides + constantes + redondantes
print("colonnes inutilisables, à supprimer en phase de nettoyage :", a_supprimer)
df = df.drop(columns=a_supprimer)  # en mémoire seulement : les fichiers CSV ne sont pas modifiés
df.shape

colonnes vides à 100 % de NaN : []
colonnes constantes (1 seule valeur renseignée) : ['Balance of Payments Manual in use']
'2-alpha code' == 'WB-2 code' dans 97% des lignes
colonnes inutilisables, à supprimer en phase de nettoyage : ['Balance of Payments Manual in use', 'WB-2 code']


(241, 29)

**Lecture** : 2 colonnes à supprimer en phase de nettoyage (31 → 29) : `Balance of Payments Manual in use` (constante) et `WB-2 code` (copie de `2-alpha code` dans 97 % des lignes). Aucune colonne vide.


### 6. Statistiques des colonnes numériques

Que disent les colonnes numériques ?


In [11]:
df.select_dtypes(include="number").describe()

,National accounts reference year,Latest industrial data,Latest trade data
count,32.00000,107.000000,185.000000
mean,2001.53125,2008.102804,2010.994595
std,5.24856,2.616834,2.569675
min,1987.00000,2000.000000,1995.000000
25%,1996.75000,2007.500000,2011.000000
50%,2002.00000,2009.000000,2012.000000
75%,2005.00000,2010.000000,2012.000000
max,2012.00000,2010.000000,2012.000000


**Lecture** : 3 colonnes numériques seulement, toutes des années de dernière donnée. Rien à en tirer pour classer les pays.


### 7. Les colonnes catégorielles

Combien de valeurs distinctes contient chaque colonne texte ?


In [12]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Other groups,2,0.8
Vital registration complete,2,0.8
IMF data dissemination standard,2,0.8
Government Accounting concept,2,0.8
System of trade,2,0.8
SNA price valuation,2,0.8
External debt Reporting status,3,1.2
PPP survey year,3,1.2
System of National Accounts,3,1.2
Lending category,3,1.2


**Lecture** : deux groupes nets, les vraies catégories (2 à 7 valeurs) d'un côté, le texte quasi unique de l'autre (20 valeurs et plus) ; n'importe quel seuil entre les deux fait le même tri.

Que contiennent ces catégories ?


In [13]:
SEUIL = 10  # n'importe quelle valeur entre 8 et 19 donne le même découpage (trou du tableau d'unicité)
cats = [col for col in cols_texte if df[col].nunique() <= SEUIL]

lignes = []
for col in cats:
    for valeur, effectif in df[col].value_counts().items():  # les valeurs existantes, hors cellules vides
        lignes.append({"colonne": col, "valeur": valeur, "effectif": effectif})
inventaire = pd.DataFrame(lignes)
inventaire

,colonne,valeur,effectif
0,Region,Europe & Central Asia,57
1,Region,Sub-Saharan Africa,48
2,Region,Latin America & Caribbean,41
3,Region,East Asia & Pacific,36
4,Region,Middle East & North Africa,21
5,Region,South Asia,8
6,Region,North America,3
7,Income Group,Upper middle income,55
8,Income Group,Lower middle income,50
9,Income Group,High income: nonOECD,44


In [14]:
sans_region = df.loc[df["Region"].isna()]
# Gibraltar et Nauru repérés à la lecture des noms : aucun critère du fichier ne sépare pays et agrégats
pays_sans_region = sans_region.loc[sans_region["Country Code"].isin(["GIB", "NRU"]), ["Country Code", "Short Name"]]
print(f"{len(sans_region)} lignes sans Region : {len(sans_region) - len(pays_sans_region)} agrégats et {len(pays_sans_region)} vrais pays.")
pays_sans_region

27 lignes sans Region : 25 agrégats et 2 vrais pays.


,Country Code,Short Name
78,GIB,Gibraltar
166,NRU,Nauru


**Lecture** : 12 colonnes catégorielles de 2 à 7 valeurs. `Region` (région) et `Income Group` (groupe de revenu) classent les 214 pays qui ont une région ; les 27 lignes restantes sont des agrégats (World, EU…), plus Gibraltar et Nauru, 2 petits pays non classés. Pour chercher des cibles, on gardera les lignes avec une région.


### Bilan EdStatsCountry

Clé validée : 241 lignes, 241 codes uniques, 0 doublon ; la jointure est sûre. 2 colonnes à supprimer en phase de nettoyage, aucune vide.

Pour academy : `Region` (région) et `Income Group` (groupe de revenu) serviront à regrouper et filtrer les pays.


## EdStatsSeries.csv : le dictionnaire des indicateurs

*Une ligne = un indicateur (ex. `SE.SEC.ENRR`, le taux de scolarisation dans le secondaire).*


### 1. Une ligne = un indicateur

Même question que pour les pays : la clé `Series Code` est-elle unique ?


In [15]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsSeries.csv")

cle = "Series Code"  # clé métier (PK) d'après le dictionnaire de 00_data_overview.ipynb
print(f"{len(df)} lignes, {df[cle].nunique()} valeurs distinctes de '{cle}'")
df[cle].is_unique  # True = chaque valeur de la clé n'apparaît que sur une seule ligne

3665 lignes, 3665 valeurs distinctes de 'Series Code'


True

**Lecture** : 3665 lignes, 3665 codes distincts. Clé unique, la jointure ne créera pas de doublons.


### 2. Taille du fichier

Combien de lignes, combien de colonnes, et de quel type ?


In [16]:
df.shape
df.info()  # en plus de shape : type et remplissage de chaque colonne

<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 20 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   str    
 1   Topic                                3665 non-null   str    
 2   Indicator Name                       3665 non-null   str    
 3   Short definition                     2156 non-null   str    
 4   Long definition                      3665 non-null   str    
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     str    
 7   Base Period                          314 non-null    str    
 8   Other notes                          552 non-null    str    
 9   Aggregation method                   47 non-null     str    
 10  Limitations and exceptions           14 non-null     str    
 11  Notes from original source           0 no

**Lecture** : 3665 lignes × 20 colonnes : 20 renseignements par indicateur.


### 3. Des doublons ?

Le même indicateur apparaît-il deux fois ?


In [17]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"{nb_doublons} doublon(s) détecté(s), à supprimer en phase de nettoyage avec drop_duplicates()")
else:
    print("aucun doublon : rien à supprimer en phase de nettoyage")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : rien à supprimer en phase de nettoyage


**Lecture** : 0 doublon, aucun indicateur en double.


### 4. Valeurs manquantes et taux de remplissage

Quelles colonnes sont remplies, et à quel point ?


In [18]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

5 colonne(s) à 100 % de remplissage, 5 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
Series Code,0,0.00,100.00
Source,0,0.00,100.00
Indicator Name,0,0.00,100.00
Long definition,0,0.00,100.00
Topic,0,0.00,100.00
Short definition,1509,41.17,58.83
Other notes,3113,84.94,15.06
Base Period,3351,91.43,8.57
Related source links,3450,94.13,5.87
Periodicity,3566,97.30,2.70


**Lecture** : 5 colonnes complètes (`Series Code`, `Indicator Name`, `Topic` (thème), `Long definition` (définition longue) et `Source`), 5 entièrement vides ; entre les deux, `Short definition` (définition courte) plafonne à 58,8 %, plus rien ne dépasse 16 %.


### 5. Colonnes inutilisables (à supprimer en phase de nettoyage)

Quelles colonnes ne distinguent pas deux indicateurs : vides à 100 %, ou constantes ?


In [19]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> rien pour distinguer les lignes
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

a_supprimer = vides + constantes
print("colonnes inutilisables, à supprimer en phase de nettoyage :", a_supprimer)
df = df.drop(columns=a_supprimer)  # en mémoire seulement : les fichiers CSV ne sont pas modifiés
df.shape

colonnes vides à 100 % de NaN : ['Unit of measure', 'Notes from original source', 'Other web links', 'Related indicators', 'License Type']
colonnes constantes (1 seule valeur renseignée) : ['Periodicity', 'Development relevance', 'Related source links']
colonnes inutilisables, à supprimer en phase de nettoyage : ['Unit of measure', 'Notes from original source', 'Other web links', 'Related indicators', 'License Type', 'Periodicity', 'Development relevance', 'Related source links']


(3665, 12)

**Lecture** : 8 colonnes à supprimer en phase de nettoyage (20 → 12) : 5 vides (`Unit of measure`, `License Type`…) et 3 constantes (`Periodicity`, `Development relevance`, `Related source links`).


### 6. Statistiques des colonnes numériques

Y a-t-il des colonnes numériques ?


In [20]:
# après avoir écarté en mémoire les colonnes identifiées au point 5 : que des colonnes texte
df.info(verbose=False)

# et dans le CSV brut ? pandas met float64 sur une colonne entièrement vide
brut = load_csv_analysis("EdStatsSeries.csv")
brut.info()

<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Columns: 12 entries, Series Code to Statistical concept and methodology
dtypes: str(12)
memory usage: 343.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 20 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Series Code                          3665 non-null   str    
 1   Topic                                3665 non-null   str    
 2   Indicator Name                       3665 non-null   str    
 3   Short definition                     2156 non-null   str    
 4   Long definition                      3665 non-null   str    
 5   Unit of measure                      0 non-null      float64
 6   Periodicity                          99 non-null     str    
 7   Base Period                          314 non-null    str    
 8   Other notes                          552 non-null    str    
 9  

**Lecture** : 0 colonne numérique une fois le point 5 écarté ; dans le CSV brut, les 5 colonnes numériques étaient exactement les vides du point 5. Un dictionnaire, c'est du texte.


### 7. Les colonnes catégorielles

Combien de valeurs distinctes contient chaque colonne texte ?


In [21]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Statistical concept and methodology,2,0.1
Aggregation method,3,0.1
Base Period,4,0.1
General comments,8,0.2
Limitations and exceptions,9,0.2
Other notes,14,0.4
Source,31,0.8
Topic,37,1.0
Short definition,1169,31.9
Long definition,2060,56.2


**Lecture** : les colonnes quasi uniques (codes, noms, définitions) d'un côté, les vraies catégories de 2 à 37 valeurs de l'autre. Seules `Topic` (thème) et `Source` sont complètes, les 6 autres catégories restent sous 16 % de remplissage (point 4).

Que contiennent ces deux catégories ?


In [22]:
df["Topic"].value_counts().head(10)

Topic
Learning Outcomes                      1046
Attainment                              733
Education Equality                      426
Secondary                               256
Primary                                 248
Population                              213
Tertiary                                158
Teachers                                137
Expenditures                             93
Engaging the Private Sector (SABER)      51
Name: count, dtype: int64

**Lecture** : 37 thèmes, dominés par l'éducation : `Learning Outcomes` (résultats d'apprentissage) et `Attainment` (niveaux atteints) pèsent ensemble près de la moitié du dictionnaire. `Topic` (thème) servira à filtrer les indicateurs pour academy.


In [23]:
df["Source"].value_counts().head(5)

Source
UNESCO Institute for Statistics                                                                         1269
Early Grade Reading Assessment (EGRA): https://www.eddataglobal.org/reading/                             403
Robert J. Barro and Jong-Wha Lee: http://www.barrolee.com/                                               360
Wittgenstein Centre for Demography and Global Human Capital: http://www.oeaw.ac.at/vid/dataexplorer/     308
Systems Approach for Better Education Results (SABER), World Bank                                        215
Name: count, dtype: int64

**Lecture** : 31 producteurs, l'UNESCO Institute for Statistics en tête (1269 indicateurs). Les autres catégories sont trop peu remplies pour servir (point 4).


### Bilan EdStatsSeries

Clé validée : 3665 lignes, 3665 codes uniques ; dans `EdStatsData.csv`, elle s'appelle `Indicator Code`. 8 colonnes à supprimer en phase de nettoyage, aucune colonne numérique.

Pour academy : `Topic` (thème) et ses 37 thèmes, dominés par l'éducation, serviront à choisir les indicateurs.


## EdStatsData.csv : les mesures par pays et indicateur

*Une ligne = la série d'un pays pour un indicateur (ex. `FRA` × `SE.SEC.ENRR`), une colonne par année de 1970 à 2100.*


### 1. Une ligne = un pays × un indicateur

Le couple `Country Code` + `Indicator Code` est-il unique, et tous les couples possibles sont-ils présents ?


In [24]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsData.csv")

# clé métier d'après le dictionnaire de 00_data_overview.ipynb : deux colonnes ensemble
cle = ["Country Code", "Indicator Code"]
print(f"{len(df)} lignes, {df.duplicated(subset=cle).sum()} doublon(s) sur cette clé")
print(f"{df['Country Code'].nunique()} pays × {df['Indicator Code'].nunique()} indicateurs = "
      f"{df['Country Code'].nunique() * df['Indicator Code'].nunique()}")
len(df) == df["Country Code"].nunique() * df["Indicator Code"].nunique()  # True = rectangle complet, aucun couple manquant

886930 lignes, 0 doublon(s) sur cette clé
242 pays × 3665 indicateurs = 886930


True

**Lecture** : 242 pays × 3665 indicateurs = 886 930 lignes, chaque couple exactement une fois : clé unique, rectangle complet.


### 2. Taille du fichier

Combien de lignes, combien de colonnes ?


In [25]:
annees = df.columns[4:]  # tout sauf les 4 colonnes d'identification
print(f"{len(annees)} colonnes d'années, de {annees[0]} à {annees[-1]}")
print(f"dont {sum(int(a) <= 2017 for a in annees)} annuelles (1970 à 2017) "
      f"et {sum(int(a) >= 2020 for a in annees)} un pas de 5 ans (2020 à 2100)")
print(df.shape)
df.info(verbose=False)  # résumé global : lister les 69 colonnes une par une serait trop long
list(annees[-20:])  # les 20 dernières colonnes : le saut 2017 -> 2020 -> 2025 se voit

65 colonnes d'années, de 1970 à 2100
dont 48 annuelles (1970 à 2017) et 17 un pas de 5 ans (2020 à 2100)
(886930, 69)
<class 'pandas.DataFrame'>
RangeIndex: 886930 entries, 0 to 886929
Columns: 69 entries, Country Name to 2100
dtypes: float64(65), str(4)
memory usage: 466.9 MB


['2015',
 '2016',
 '2017',
 '2020',
 '2025',
 '2030',
 '2035',
 '2040',
 '2045',
 '2050',
 '2055',
 '2060',
 '2065',
 '2070',
 '2075',
 '2080',
 '2085',
 '2090',
 '2095',
 '2100']

**Lecture** : 886 930 lignes × 69 colonnes : 4 colonnes d'identification, puis 65 colonnes d'années, chaque année de 1970 à 2017 puis une tous les 5 ans jusqu'en 2100 (projections comprises).


### 3. Des doublons ?

Une même série pays-indicateur est-elle mesurée deux fois ?


In [26]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"{nb_doublons} doublon(s) détecté(s), à supprimer en phase de nettoyage avec drop_duplicates()")
else:
    print("aucun doublon : rien à supprimer en phase de nettoyage")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : rien à supprimer en phase de nettoyage


**Lecture** : 0 doublon, aucune série en double.


### 4. Valeurs manquantes et taux de remplissage

Quelles années sont remplies ? Table : les mieux remplies et les plus vides.


In [27]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

identification = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]
print(f"identification complète : {(remplissage.loc[identification, "taux_remplissage_%"] == 100).all()}")

taux_annees = remplissage.loc[remplissage.index.difference(identification), "taux_remplissage_%"]
print(f"remplissage des 65 années : de {taux_annees.min():.2f} % ({taux_annees.idxmin()}) à {taux_annees.max():.2f} % ({taux_annees.idxmax()})")

pd.concat([remplissage.head(), remplissage.tail()])  # les mieux remplies + les plus vides

identification complète : True
remplissage des 65 années : de 0.02 % (2017) à 27.33 % (2010)


,nb_manquants,proportion_manquante_%,taux_remplissage_%
Country Name,0,0.00,100.00
Country Code,0,0.00,100.00
Indicator Name,0,0.00,100.00
Indicator Code,0,0.00,100.00
2010,644488,72.67,27.33
1972,851311,95.98,4.02
1973,851385,95.99,4.01
1971,851393,95.99,4.01
2016,870470,98.14,1.86
2017,886787,99.98,0.02


**Lecture** : les 4 colonnes d'identification sont complètes ; les années au mieux au quart : 27,33 % en 2010, quand 2017 s'effondre à 0,02 %. Aucune année entièrement vide : chaque indicateur est collecté sur ses propres années d'enquête.


### 5. Colonnes inutilisables (à supprimer en phase de nettoyage)

Des colonnes vides ou constantes à écarter ? Rappel : les années sont l'axe du temps, elles ne sont pas concernées, et une série qui garde la même valeur d'année en année reste une vraie mesure.


In [28]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) attributs constants : une seule valeur pour tout le fichier,
#    rien pour distinguer les lignes. Les années sont l'axe du temps,
#    on ne les teste pas (voir la cellule suivante)
attributs = [c for c in df.columns if c not in annees]
constantes = [c for c in attributs if df[c].nunique() == 1]
print("attributs constants (1 seule valeur) :", constantes)

print("colonnes inutilisables :", vides + constantes)
df.shape  # inchangé : rien à supprimer

colonnes vides à 100 % de NaN : []


attributs constants (1 seule valeur) : []
colonnes inutilisables : []


(886930, 69)

In [29]:
# la stabilité dans le temps se lit en ligne : un pays × indicateur peut
# garder la même valeur chaque année, c'est une vraie mesure, on la garde
remplis = df[annees].notna().sum(axis=1)     # nb d'années remplies par série
distincts = df[annees].nunique(axis=1)        # nb de valeurs différentes par série
n_stables = ((remplis >= 10) & (distincts == 1)).sum()
print(f"{n_stables} séries gardent la même valeur sur 10 années ou plus : toutes conservées")

5528 séries gardent la même valeur sur 10 années ou plus : toutes conservées


**Lecture** : première section sans colonne à supprimer : aucune vide, aucune constante, même 2017 (0,02 %) garde des mesures ; 5 528 séries gardent la même valeur sur 10 années ou plus, de vraies mesures, elles restent. Le tri des années utiles se fera à l'analyse.


### 6. Statistiques des colonnes numériques

Une colonne = une année, et chaque année mélange tous les indicateurs : que valent les statistiques ? Exemple sur 3 années.


In [30]:
stats = df[annees].describe().T  # transposé : une ligne par année

print(f"valeurs mesurées : de {stats['count'].min():.0f} ({stats['count'].idxmin()}) "
      f"à {stats['count'].max():.0f} ({stats['count'].idxmax()})")

proj = [a for a in annees if int(a) >= 2020]  # les colonnes de projection
n_porteurs = df.loc[df[proj[0]].notna(), "Indicator Code"].nunique()
print(f"les {len(proj)} colonnes de projection ({proj[0]}-{proj[-1]}) : "
      f"{stats.loc[proj, 'count'].unique()} valeurs chacune, portées par {n_porteurs} indicateurs")

# toute la table ne se lit pas : chaque année mélange des pourcentages
# et des populations. 3 années suffisent à le voir :
stats.loc[["1970", "2010", "2017"]]

valeurs mesurées : de 143 (2017) à 242442 (2010)
les 17 colonnes de projection (2020-2100) : [51436.] valeurs chacune, portées par 308 indicateurs


,count,mean,std,min,25%,50%,75%,max
1970,72288.0,1.974772e+09,1.211687e+11,-1.435564,0.890000,6.317724,62.5125,1.903929e+13
2010,242442.0,1.189592e+10,6.218710e+11,-496905.000000,1.322703,20.460199,3121.0000,9.134676e+13
2017,143.0,2.160839e+00,8.105845e-01,1.000000,2.000000,2.000000,3.0000,4.000000e+00


**Lecture** : ce tableau ne veut rien dire : la moyenne mélange pourcentages et populations (11,9 milliards en 2010, 2,16 en 2017). À retenir : de 143 valeurs mesurées en 2017 à 242 442 en 2010 ; les 17 années de projection (2020 à 2100) portent chacune 51 436 valeurs sur 308 indicateurs ; des valeurs négatives existent (jusqu'à -496 905).

Les comparaisons n'auront de sens qu'indicateur par indicateur : c'est le travail de l'analyse.


### 7. Les colonnes catégorielles

Les 4 colonnes texte donnent l'identité de chaque ligne : sont-elles cohérentes avec les deux référentiels ?


In [31]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Country Name,242,0.0
Country Code,242,0.0
Indicator Name,3665,0.4
Indicator Code,3665,0.4


**Lecture** : 4 colonnes quasi uniques : le nom et le code du pays se répondent (242), le nom et le code de l'indicateur aussi (3665). Pas de vraie catégorie à compter ici.

Ces codes existent-ils tous dans les référentiels ?


In [32]:
# les codes mesurés doivent exister dans le référentiel des pays
# et dans le dictionnaire des indicateurs
pays = load_csv_analysis("EdStatsCountry.csv")   # référentiel des pays
indic = load_csv_analysis("EdStatsSeries.csv")   # dictionnaire des indicateurs

# les codes sont tapés à la main dans les deux fichiers : on les met au
# propre (espaces en trop, majuscules) avant de comparer, sinon des
# indicateurs semblent absents du dictionnaire
df["Indicator Code"] = df["Indicator Code"].str.strip().str.upper()
indic["Series Code"] = indic["Series Code"].str.strip().str.upper()

codes_indic = set(df["Indicator Code"])
dico = set(indic["Series Code"])
print(f"{len(codes_indic - dico)} indicateur(s) mesuré(s) absent(s) du dictionnaire")
print(f"{len(dico - codes_indic)} indicateur(s) du dictionnaire sans aucune mesure")
print(f"doublons sur la clé après mise au propre : "
      f"{df.duplicated(subset=['Country Code', 'Indicator Code']).sum()}")

codes_pays = set(df["Country Code"])
connus = set(pays["Country Code"])
orphelins_pays = sorted(codes_pays - connus)
print(f"{len(connus - codes_pays)} pays du référentiel sans aucune mesure")
print(f"{len(orphelins_pays)} pays mesuré(s) sans fiche au référentiel : {orphelins_pays}")

# et pourtant, ces codes orphelins ont bien des mesures (souvent ONU ou
# UIS, qui couvrent aussi les territoires non souverains)
for code in orphelins_pays:
    n_indic = df.loc[df["Country Code"] == code, annees].notna().any(axis=1).sum()
    print(f"{code} : {n_indic} indicateur(s) avec au moins une valeur")

# un pays sans fiche ne peut pas être classé (ni région, ni groupe de
# revenu) : ses lignes seront écartées en phase de nettoyage
lignes_gardees = df.loc[~df["Country Code"].isin(orphelins_pays)]
print(f"{len(lignes_gardees)} lignes gardées si on écarte {orphelins_pays} en phase de nettoyage")

0 indicateur(s) mesuré(s) absent(s) du dictionnaire
0 indicateur(s) du dictionnaire sans aucune mesure
doublons sur la clé après mise au propre : 0
0 pays du référentiel sans aucune mesure
1 pays mesuré(s) sans fiche au référentiel : ['VGB']
VGB : 138 indicateur(s) avec au moins une valeur
883265 lignes gardées si on écarte ['VGB'] en phase de nettoyage


**Lecture** : tous les indicateurs mesurés existent dans le dictionnaire, la clé reste unique. Un seul pays mesuré sans fiche : `VGB` (Îles Vierges britanniques) ; sans région ni groupe de revenu, ses lignes ne peuvent pas être classées : la phase de nettoyage les écartera, il resterait 883 265 lignes, toutes joignables.


### Bilan EdStatsData

Clé validée : 886 930 lignes, rectangle complet, aucun doublon, aucune colonne à supprimer. `VGB` écarté, il resterait 883 265 lignes joignables aux deux référentiels.

Pour Mark : les mesures se concentrent sur les années récentes (au mieux 27,33 % en 2010), et les projections 2020-2100 ne portent que 308 indicateurs.


## EdStatsCountry-Series.csv : les sources par pays et indicateur

*Une ligne = la source d'un couple pays × indicateur (ex. `FRA` × `SP.POP.TOTL`).*


### 1. Une ligne = un pays × un indicateur

Le couple `CountryCode` + `SeriesCode` est-il unique ?


In [33]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsCountry-Series.csv")

# clé métier d'après le dictionnaire de 00_data_overview.ipynb : deux colonnes ensemble
cle = ["CountryCode", "SeriesCode"]
print(f"{len(df)} lignes, {df.duplicated(subset=cle).sum()} doublon(s) sur cette clé")
len(df.drop_duplicates(subset=cle)) == len(df)  # True = chaque couple est sur une seule ligne

613 lignes, 0 doublon(s) sur cette clé


True

**Lecture** : 613 lignes, aucun couple en double. Clé unique.


### 2. Taille du fichier

Combien de lignes, combien de colonnes ?


In [34]:
df.shape
df.info()  # en plus de shape : type et remplissage de chaque colonne

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   CountryCode  613 non-null    str  
 1   SeriesCode   613 non-null    str  
 2   DESCRIPTION  613 non-null    str  
dtypes: str(3)
memory usage: 14.5 KB


**Lecture** : 613 lignes × 3 colonnes, une ligne par couple documenté.


### 3. Des doublons ?

Le même couple est-il documenté deux fois ?


In [35]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"{nb_doublons} doublon(s) détecté(s), à supprimer en phase de nettoyage avec drop_duplicates()")
else:
    print("aucun doublon : rien à supprimer en phase de nettoyage")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : rien à supprimer en phase de nettoyage


**Lecture** : 0 doublon, aucune source en double.


### 4. Valeurs manquantes et taux de remplissage

Quelles colonnes sont remplies ?


In [36]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

3 colonne(s) à 100 % de remplissage, 0 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
CountryCode,0,0.0,100.0
SeriesCode,0,0.0,100.0
DESCRIPTION,0,0.0,100.0


**Lecture** : les 3 colonnes remplies à 100 %, aucune valeur manquante.


### 5. Colonnes inutilisables (à supprimer en phase de nettoyage)

Des colonnes vides ou constantes à écarter ?


In [37]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> rien pour distinguer les lignes
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

print("colonnes inutilisables :", vides + constantes)
df.shape  # inchangé : rien à supprimer

colonnes vides à 100 % de NaN : []
colonnes constantes (1 seule valeur renseignée) : []
colonnes inutilisables : []


(613, 3)

**Lecture** : aucune, les 3 colonnes restent : `CountryCode` et `SeriesCode` pour les jointures, `DESCRIPTION` pour la source de la mesure.


### 6. Statistiques des colonnes numériques

Y a-t-il des colonnes numériques ?


In [38]:
df.info(verbose=False)  # la ligne "dtypes" donne le nombre de colonnes de chaque type

<class 'pandas.DataFrame'>
RangeIndex: 613 entries, 0 to 612
Columns: 3 entries, CountryCode to DESCRIPTION
dtypes: str(3)
memory usage: 14.5 KB


**Lecture** : 0 colonne numérique, les 3 colonnes sont du texte. Tout se joue au point 7.


### 7. Les colonnes catégorielles

Combien de valeurs distinctes dans chaque colonne ?


In [39]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
SeriesCode,21,3.4
DESCRIPTION,97,15.8
CountryCode,211,34.4


**Lecture** : 211 pays et 21 indicateurs documentés ; `DESCRIPTION`, avec 97 textes différents, porte l'information du fichier.

Qui sont ces pays, et ces indicateurs ?


In [40]:
comptes_pays = df["CountryCode"].value_counts()
print(f"de {comptes_pays.min()} à {comptes_pays.max()} ligne(s) par pays")
comptes_pays.head(10)

de 2 à 18 ligne(s) par pays


CountryCode
GEO    18
MDA    18
CYP    12
MAR    12
MUS    12
SRB    12
TZA    12
AFG     8
ARG     8
ERI     8
Name: count, dtype: int64

**Lecture** : de 2 à 18 lignes par pays, la Géorgie (`GEO`) et la Moldavie (`MDA`) en tête (18 lignes chacune).

Et du côté des indicateurs ?


In [41]:
comptes_indic = df["SeriesCode"].value_counts()
print(f"de {comptes_indic.min()} à {comptes_indic.max()} ligne(s) par indicateur")
comptes_indic

de 4 à 211 ligne(s) par indicateur


SeriesCode
SP.POP.TOTL          211
SP.POP.GROW          211
NY.GDP.PCAP.PP.CD     19
NY.GDP.PCAP.PP.KD     19
NY.GNP.PCAP.PP.CD     19
NY.GDP.MKTP.PP.KD     14
NY.GNP.MKTP.PP.CD     14
NY.GDP.MKTP.PP.CD     14
SP.POP.1564.TO.ZS     13
SP.POP.TOTL.MA.ZS     13
SP.POP.TOTL.FE.ZS     13
SP.POP.0014.TO.ZS     13
NY.GNP.PCAP.CD         6
NY.GDP.PCAP.CD         5
NY.GDP.PCAP.KD         5
SP.POP.1564.MA.IN      4
SP.POP.0014.TO         4
SP.POP.1564.TO         4
SP.POP.1564.FE.IN      4
SP.POP.0014.MA.IN      4
SP.POP.0014.FE.IN      4
Name: count, dtype: int64

**Lecture** : deux indicateurs couvrent les 211 pays : `SP.POP.TOTL` (population totale) et `SP.POP.GROW` (croissance de la population) ; les 19 autres vont de 4 à 19 lignes.

Que disent les textes de `DESCRIPTION` ?


In [42]:
df["DESCRIPTION"].value_counts().head(10)

DESCRIPTION
Data sources : United Nations World Population Prospects                                       154
Data sources: United Nations World Population Prospects                                        137
Estimates are based on regression.                                                              84
Data sources : Eurostat                                                                         54
Derived using ratio of age group from WPP and World Bank population.                            24
Data are for areas under the effective control of the Government of the Republic of Cyprus.      6
Excludes Abkhazia and South Ossetia.                                                             6
Includes Former Spanish Sahara.                                                                  6
Excludes Transnistria.                                                                           6
Covers mainland Tanzania only.                                                                   

**Lecture** : la source la plus citée est « United Nations World Population Prospects », en deux écritures (154 et 137 lignes). Ces textes disent d'où viennent les chiffres, ils ne serviront pas à choisir les pays.


### Bilan EdStatsCountry-Series

Clé validée : 613 lignes, chaque couple exactement une fois, aucune valeur manquante. Un petit fichier complet.

Il documente surtout la population (`SP.POP.TOTL` et `SP.POP.GROW` sur les 211 pays). Pour choisir les pays, il ne servira pas directement : il dit d'où viennent les chiffres.


## EdStatsFootNote.csv : les annotations par pays, indicateur et année

*Une ligne = une annotation pour un pays ou agrégat × un indicateur × une année (ex. `ABW` × `SE.PRE.ENRL.FE` × `YR2001`).*


### 1. Une ligne = un pays × un indicateur × une année

Le triplet `CountryCode` + `SeriesCode` + `Year` est-il unique ?


In [43]:
# DataFrame dédié au calcul pour cette section : on recharge le CSV brut,
# séparément du `dfs` réservé à l'affichage
df = load_csv_analysis("EdStatsFootNote.csv")

# clé métier d'après le dictionnaire de 00_data_overview.ipynb : trois colonnes ensemble
cle = ["CountryCode", "SeriesCode", "Year"]
print(f"{len(df)} lignes, {df.duplicated(subset=cle).sum()} doublon(s) sur cette clé")
len(df.drop_duplicates(subset=cle)) == len(df)  # True = chaque triplet est sur une seule ligne

643638 lignes, 0 doublon(s) sur cette clé


True

**Lecture** : 643 638 lignes, aucun triplet en double. Clé unique.


### 2. Taille du fichier

Combien de lignes, combien de colonnes ?


In [44]:
df.shape
df.info()  # en plus de shape : type et remplissage de chaque colonne

<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Data columns (total 4 columns):
 #   Column       Non-Null Count   Dtype
---  ------       --------------   -----
 0   CountryCode  643638 non-null  str  
 1   SeriesCode   643638 non-null  str  
 2   Year         643638 non-null  str  
 3   DESCRIPTION  643638 non-null  str  
dtypes: str(4)
memory usage: 19.6 MB


**Lecture** : 643 638 lignes × 4 colonnes, le deuxième fichier du lot derrière `EdStatsData.csv`.


### 3. Des doublons ?

La même annotation est-elle répétée deux fois ?


In [45]:
nb_doublons = df.duplicated().sum()
print(f"{nb_doublons} ligne(s) en double (toutes colonnes confondues)")
if nb_doublons > 0:
    df = df.drop_duplicates()
    print(f"{nb_doublons} doublon(s) détecté(s), à supprimer en phase de nettoyage avec drop_duplicates()")
else:
    print("aucun doublon : rien à supprimer en phase de nettoyage")

0 ligne(s) en double (toutes colonnes confondues)
aucun doublon : rien à supprimer en phase de nettoyage


**Lecture** : 0 doublon, aucune annotation en double.


### 4. Valeurs manquantes et taux de remplissage

Quelles colonnes sont remplies ?


In [46]:
remplissage = pd.DataFrame({
    "nb_manquants": df.isnull().sum(),
    "proportion_manquante_%": (df.isnull().mean() * 100).round(2),
    "taux_remplissage_%": ((1 - df.isnull().mean()) * 100).round(2),
}).sort_values("taux_remplissage_%", ascending=False)  # les mieux remplies en premier

n_completes = (remplissage["taux_remplissage_%"] == 100).sum()  # 100 % = aucune valeur manquante
n_vides = (remplissage["taux_remplissage_%"] == 0).sum()        # 0 % = colonne entièrement vide
print(f"{n_completes} colonne(s) à 100 % de remplissage, {n_vides} colonne(s) vide(s) à 0 %")

remplissage

4 colonne(s) à 100 % de remplissage, 0 colonne(s) vide(s) à 0 %


,nb_manquants,proportion_manquante_%,taux_remplissage_%
CountryCode,0,0.0,100.0
SeriesCode,0,0.0,100.0
Year,0,0.0,100.0
DESCRIPTION,0,0.0,100.0


**Lecture** : les 4 colonnes remplies à 100 %, aucune valeur manquante.


### 5. Colonnes inutilisables (à supprimer en phase de nettoyage)

Des colonnes vides ou constantes à écarter ?


In [47]:
# 1) colonnes vides à 100 % de NaN
vides = df.columns[df.isnull().all()].tolist()
print("colonnes vides à 100 % de NaN :", vides)

# 2) colonnes constantes : une seule valeur renseignée -> rien pour distinguer les lignes
constantes = [c for c in df.columns if df[c].nunique() == 1]
print("colonnes constantes (1 seule valeur renseignée) :", constantes)

print("colonnes inutilisables :", vides + constantes)
df.shape  # inchangé : rien à supprimer

colonnes vides à 100 % de NaN : []


colonnes constantes (1 seule valeur renseignée) : []
colonnes inutilisables : []


(643638, 4)

**Lecture** : aucune, les 4 colonnes restent : trois identifient la mesure commentée, la quatrième porte l'annotation.


### 6. Statistiques des colonnes numériques

Y a-t-il des colonnes numériques ?


In [48]:
df.info(verbose=False)  # la ligne "dtypes" donne le nombre de colonnes de chaque type
df["Year"].head(3)  # l'année telle qu'elle est stockée : "YR2001", donc du texte

<class 'pandas.DataFrame'>
RangeIndex: 643638 entries, 0 to 643637
Columns: 4 entries, CountryCode to DESCRIPTION
dtypes: str(4)
memory usage: 19.6 MB


0    YR2001
1    YR2005
2    YR2000
Name: Year, dtype: str

**Lecture** : 0 colonne numérique : les 4 colonnes sont du texte, `Year` (l'année) comprise, stockée `YR2001` et non 2001. Les comptages se font au point 7.


### 7. Les colonnes catégorielles

Combien de valeurs distinctes dans chaque colonne ?


In [49]:
cols_texte = df.select_dtypes(exclude="number").columns

unicite = pd.DataFrame({
    "nb_valeurs_distinctes": [df[c].nunique() for c in cols_texte],
    "part_des_lignes_%": [round(df[c].nunique() / len(df) * 100, 1) for c in cols_texte],
}, index=cols_texte).sort_values("nb_valeurs_distinctes")
unicite

,nb_valeurs_distinctes,part_des_lignes_%
Year,56,0.0
CountryCode,239,0.0
SeriesCode,1558,0.2
DESCRIPTION,9102,1.4


**Lecture** : 239 codes pays ou agrégats, 1 558 indicateurs, 56 écritures d'années, 9 102 annotations différentes.

Que disent les années, les pays et les annotations ?


In [50]:
df["Year"].value_counts().head(10)

Year
YR2004    27128
YR2005    25992
YR2002    25687
YR2003    25683
YR2000    25093
YR2001    25047
YR2008    25018
YR1999    24938
YR2007    24263
YR2006    23394
Name: count, dtype: int64

**Lecture** : les années les plus annotées vont de 1999 à 2008, `YR2004` en tête (27 128 lignes) ; chaque année est écrite avec le préfixe `YR`.

Et les codes pays ?


In [51]:
df["CountryCode"].value_counts().head(10)

CountryCode
LIC    7320
CYP    7183
LDC    6481
SSA    6389
SSF    6336
HPC    6237
LAC    6116
MIC    5862
LMY    5839
LCN    5779
Name: count, dtype: int64

**Lecture** : 239 codes annotés ; le top 10 mélange groupes de pays (`LIC`, `LDC`, `SSA`) et pays seuls, comme Chypre (`CYP`, 7 183 lignes). Le fichier documente les deux types sans distinction.


In [52]:
df["SeriesCode"].value_counts().head(10)

SeriesCode
SH.DYN.MORT          9226
SE.PRM.AGES          8771
SE.PRM.DURS          8771
SE.SEC.DURS          8619
SE.SEC.AGES          8581
SE.PRM.ENRL          6610
SE.PRM.ENRR          6418
SE.PRM.ENRL.FE.ZS    6177
SE.PRM.ENRL.FE       6165
SE.ENR.PRIM.FM.ZS    6013
Name: count, dtype: int64

**Lecture** : 1 558 indicateurs annotés, le plus annoté étant `SH.DYN.MORT` avec 9 226 lignes.


In [53]:
df["DESCRIPTION"].value_counts().head(10)

DESCRIPTION
Country Data                                      191188
UNESCO Institute for Statistics (UIS) estimate    171527
Estimated                                         117155
UIS Estimation                                     31395
Country estimation.                                26308
National Estimate                                  23570
Country data                                       16353
National Estimation                                12001
UNSD WPP08 revision file 2A and 3A.                 3480
Reference Period: 2005-2014.                        3406
Name: count, dtype: int64

**Lecture** : l'annotation la plus fréquente est `Country Data` (données du pays), avec 191 188 lignes ; plusieurs libellés signalent des estimations (`Estimated`, `UIS Estimation`, `National Estimation`), et la même idée s'écrit de deux façons (`Country Data` / `Country data`).


### Bilan EdStatsFootNote

Clé validée : 643 638 lignes, chaque triplet exactement une fois, aucune valeur manquante. Un fichier d'annotations complet.

Les annotations signalent souvent des estimations : elles aideront à repérer les valeurs à manier avec prudence dans `EdStatsData.csv`. La mise au propre (années en nombres, libellés répétés) attendra la phase de nettoyage.


## Synthèse : qualité des 5 fichiers EdStats

### Ce que couvrent les données

- **Pays** : 241 pays et agrégats, clé `Country Code` unique (section `EdStatsCountry.csv`, point 1) ; côté mesures, 242 codes, dont `VGB` sans fiche, à écarter : 883 265 lignes joignables (section `EdStatsData.csv`, point 7).
- **Indicateurs** : 3665 indicateurs en 37 thèmes dominés par l'éducation (section `EdStatsSeries.csv`, points 1 et 7).
- **Années** : 65 colonnes d'années dans les mesures, de 1970 à 2100 (section `EdStatsData.csv`, point 2).
- **Volume** : 886 930 lignes de mesures, 613 sources, 643 638 annotations ; aucun doublon dans les 5 fichiers (point 3 de chaque section).

### Leurs lacunes

- **Des manques très inégaux** : au mieux 27,33 % de remplissage pour une année (2010), et 2017 à 0,02 % (section `EdStatsData.csv`, point 4) ; dans le référentiel, `Region` (région) et `Income Group` (groupe de revenu) descendent à 88,8 % (section `EdStatsCountry.csv`, point 4).
- **Des projections** : les années 2020 à 2100 portent chacune 51 436 valeurs sur 308 indicateurs de population, ce ne sont pas des mesures (section `EdStatsData.csv`, point 6).
- **Des agrégats mêlés aux pays** : 27 lignes sans région, agrégats plus 2 petits pays non classés (section `EdStatsCountry.csv`, point 7) ; pour viser des pays, on ne gardera que les lignes avec une région.
- **Des colonnes à supprimer** en phase de nettoyage : 2 au référentiel, 8 au dictionnaire (point 5 des sections concernées).

### Suffisent-elles à guider l'expansion d'academy ?

Oui, pour une première sélection de pays cibles : clés uniques, thèmes éducatifs nombreux, mesures annuelles de 1970 à 2017.

Trois précautions :

- ne garder que les lignes avec une région, pour écarter les agrégats ;
- s'appuyer sur les années les mieux remplies, pas sur 2016 et 2017, quasi vides (section `EdStatsData.csv`, point 4) ;
- traiter les projections 2020-2100 comme des tendances, pas comme des mesures (section `EdStatsData.csv`, point 6).

Ensuite : choisir les indicateurs via `Topic` (thème), en croisant avec les annotations, qui signalent les estimations (section `EdStatsFootNote.csv`, point 7).
